# Word2Vec embedding pipeline

本轮正式方案：仅原项目训练集的完整 `words`（共享清洗、文本处理和划分编码之后；未去停用词、未词干化），不引入 WebMD，不应用短语规则。只在 train 学习词表；valid/test 只生成向量。

| 参数 | 默认值 |
|---|---|
| vector_size | 300 |
| window_size | 5 |
| min_count | 5 |
| max_iter | 10 |
| step_size | 0.025 |
| seed | 42 |
| num_partitions | 1 |
| max_sentence_length | 1000 |
| pooling | meanmax |

meanmax 将平均向量与逐维最大值拼接，输出 **600 维**。重复词重复计入；均值分母包含 OOV token，OOV 不参与最大值；空/全 OOV 评论输出零向量。`p01` 则输出原生平均词向量，维度等于 `vector_size`。

环境按仓库 README：Python 3.12 / PySpark 4.2 / Java 21，依赖来自 `requirements.txt`。选择对应的 Jupyter kernel，在外部配置 Java/Hadoop 环境，不在 notebook 写入个人机器路径。

## 1. 运行配置与会话

- `REBUILD=False`：读取已有共享 split；首次运行无缓存，或公共清洗/文本处理/编码/划分配置变化时，按团队约定改为 `True` 重建。重建会覆盖共享 split。
- `TRAIN=False`：默认加载交付的 `word2vec` 模型并重新生成评论向量；`True` 才重新训练并覆盖 `model_name` 对应的模型目录。`REBUILD` 不控制模型训练。
- `word2vec_config={}`：训练时使用模块的默认配置。自定义时只填写要覆盖的键，并设置 `TRAIN=True`；预测读取模型目录保存的配置。
- 新实验使用独立 `model_name`（小写字母、数字和单下划线，不含 `__`），避免覆盖正式模型及其向量表。

例如重新训练 200 维 meanmax：设置 `TRAIN=True`、`model_name="word2vec_custom"`、`word2vec_config={"vector_size": 200, "pooling": "meanmax"}`，输出400维。按原默认配置重训则设置 `TRAIN=True` 并保留空配置。修改已导入代码或 Java/内存设置后重启 kernel。

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src").is_dir() and (candidate / "project.ipynb").is_file():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pyspark.sql import SparkSession
from src.dataOI import DataOI
from src.paths import EMBEDDING_MODELS_DIR, EMBEDDING_DIR

SEED = 42
REBUILD = False
TRAIN = False
embedding_name = "word2vec"
model_name = "word2vec"
word2vec_config = {}

if not TRAIN and word2vec_config:
    raise ValueError("Custom config requires TRAIN=True; prediction uses the saved model config.")

spark = (SparkSession.builder
         .master("local[*]")
         .appName("Word2VecPipeline")
         .config("spark.driver.memory", "4g")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
data_oi = DataOI()

## 2. 准备共享输入

沿用公共入口的清洗、文本处理、seed=42 哈希划分和训练集拟合编码，不额外去重或变更样本。公共文本处理生成的其他列会保留在输入中，但 Word2Vec 只读取 `words`。

旧实验使用旧提交对应的输入。Git pull 不会更新已有 Parquet；旧缓存与最新代码是否匹配需单独确认。不要把此入口在新版输入上运行，直接称为历史实验的逐位复现。

In [ ]:
from src.cleaning import data_cleaning
from src.text_features import text_feature_engineering
from src.encoder import encoder
from src.split import split, weight

if REBUILD:
    df_raw = data_oi.load_raw_data()
    df_cleaned = data_cleaning(df_raw)
    df_cleaned.cache().count()
    df_engineered = text_feature_engineering(df_cleaned, True)
    df_engineered.cache().count()
    df_train, df_valid, df_test = split(df_engineered, SEED)
    df_train = weight(df_train)
    df_train, df_valid, df_test = encoder(df_train, df_valid, df_test)
    data_oi.save_split_data(df_train, df_valid, df_test)
    data_oi.unpersist(df_cleaned, df_engineered)
    del df_raw, df_cleaned, df_engineered
else:
    df_train, df_valid, df_test = data_oi.load_split_data()


## 3. 训练或复用词模型

固定注册名 `word2vec`，保存目录由 `model_name` 决定。训练调用现有注册函数，可传自定义配置；复用时不调用 train。两种方式随后都通过 predict 从磁盘读取模型及 `embedding_config.json`。不拟合或传入短语规则。

保存位置：`src/embeddings/models/<model_name>/`。训练会替换同名模型，历史 `provenance.json` 不会被当作新训练的来源记录继续沿用。

In [ ]:
from src.embeddings import EmbeddingModel

embedding_model = EmbeddingModel(df_train, df_valid, df_test)
if TRAIN:
    print(f"Training Word2Vec: {model_name}")
    EmbeddingModel.train_registry[embedding_name](
        embedding_model, model_name=model_name, config=word2vec_config,
    )
else:
    print(f"Reusing saved Word2Vec: {model_name}")

saved_config = json.loads(
    (EMBEDDING_MODELS_DIR / model_name / "embedding_config.json").read_text(encoding="utf-8")
)
print(json.dumps(saved_config, ensure_ascii=False, indent=2))

## 4. 生成并保存三份评论向量

predict 返回 train、valid、test，使用模型自身的配置聚合。输出保留仓库约定的 ID、标签、训练权重和编码特征，`embedding` 为 Spark ML `VectorUDT`；文本列由基类统一裁剪。

由现有 `DataOI` 保存到 `data/processed/embeddings/<model_name>/{train,valid,test}.parquet`，会覆盖同名向量缓存。生成缓存不提交 Git。

In [ ]:
print(f"Generating embeddings: {model_name}")
train_embedding, valid_embedding, test_embedding = (
    EmbeddingModel.predict_registry[embedding_name](embedding_model, model_name)
)
data_oi.save_embedding_data(
    model_name, train_embedding, valid_embedding, test_embedding,
)
print(f"Saved {saved_config['output_dimension']}D embeddings to {EMBEDDING_DIR / model_name}")
train_embedding.printSchema()
train_embedding.select("uniqueID", "rating", "embedding").show(3, truncate=80)

## 交接与历史记录

下游可通过 `data_oi.load_embedding_data(model_name)` 读取这三份表。本入口只完成 Word2Vec；analytics 负责人通过现有入口调用自己的注册函数。公共 analytics runner 按注册名读取 `word2vec`；自定义名字用于个人实验，需要在相应个人入口显式选择，不会新增注册项。

交付词模型按字节来自历史 `w2v_t20261108_225759_d300`：旧提交 `46b1a03` 对应原训练集77,800条、6,702,823个token，词表14,785。历史不加权 `lr_baseline` 在验证集上的 accuracy=40.7284%、weighted F1=0.355288、MAE=1.739302、RMSE=2.938868，仅作为旧输入下的记录，不代表新版管线或测试集得分；本 notebook 不执行该分类器。原生模型指纹见模型目录的 `provenance.json`，实际聚合配置见 `embedding_config.json`。重新训练后需另行记录新输入版本和评估结果。

本 notebook 尚未执行完整数据训练；代码单元格输出及执行计数保持为空。